In [ ]:
# 1. sort out data with label "MDD" and "HEALTHY"
import pandas as pd
from pathlib import Path
import os
import random
import numpy as np
import torch

SEED = 42
# Python built-in
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
# NumPy
np.random.seed(SEED)
# PyTorch
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# CuDNN
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


data_dir = Path("E:/preprocessed_data_high gamma")
label_path = Path("E:/TDBRAIN_participants_V2.xlsx")

existing_ids = [p.name for p in data_dir.iterdir() if p.is_dir()]
df = pd.read_excel(label_path)

df_filtered = df[df["participants_ID"].isin(existing_ids)]
df_filtered = df_filtered[df_filtered["formal_status"].isin(["MDD", "HEALTHY"])]

print("Total number of eligible label samples:", len(df_filtered))
print("MDD samples:", (df_filtered["formal_status"] == "MDD").sum())
print("HEALTHY samples:", (df_filtered["formal_status"] == "HEALTHY").sum())

label_dict = dict(zip(df_filtered["participants_ID"], df_filtered["formal_status"]))

In [ ]:
# 2. load EEG data -> EA + Within-subject normalization (preserve temporal sequence)
import os, random
import mne
import numpy as np
import scipy
from tqdm import tqdm

def EA_whitten(data):
    aligned_data = []
    for trials in data:
        covs = [np.cov(trial) for trial in trials]
        mean_cov = np.mean(covs, axis=0)
        mean_cov_inv_sqrt = np.linalg.pinv(scipy.linalg.sqrtm(mean_cov))
        aligned_trials = []
        for trial in trials:
            trial_aligned = mean_cov_inv_sqrt @ trial
            trial_aligned = (trial_aligned - np.mean(trial_aligned, axis=1, keepdims=True)) / \
                            (np.std(trial_aligned, axis=1, keepdims=True) + 1e-6)
            aligned_trials.append(trial_aligned)
        aligned_data.append(aligned_trials)
    return aligned_data

X_all, y_all, subject_ids = [], [], []
for subj_id, label in tqdm(label_dict.items(), desc="Loading EEG data"):
    subj_epochs = []
    for ses in range(1,5):
        p = data_dir/subj_id/f"ses-{ses}"/"eeg"/"preprocessed-epo.fif"
        if p.exists():
            try:
                epochs = mne.read_epochs(str(p), preload=True, verbose=False)
                data = epochs.get_data()
                subj_epochs.extend([trial for trial in data])
            except Exception as e:
                print(f"loading fail {p}: {e}")
    if subj_epochs:
        X_all.append(subj_epochs)
        y_all.append(0 if label=="HEALTHY" else 1)
        subject_ids.append(subj_id)

print(f"Number of successfully read subjects: {len(X_all)}")
X_all = EA_whitten(X_all)

In [ ]:
# 3. Expand each subject's epoch to construct X, y, subj_ids
X, y, subj_ids = [], [], []
for trials, label, sid in zip(X_all, y_all, subject_ids):
    X.extend(trials)
    y.extend([label]*len(trials))
    subj_ids.extend([sid]*len(trials))

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=int)
subj_ids = np.array(subj_ids)
print("total trials number：", X.shape[0], "dimension：", X.shape)


In [ ]:
# 4. Construct training/validation sets by category (HEALTHY 80/20, MDD equal amount for training) 
from sklearn.model_selection import train_test_split

idx_h = np.where(y==0)[0]
idx_m = np.where(y==1)[0]

X_h, y_h = X[idx_h], y[idx_h]
X_m, y_m = X[idx_m], y[idx_m]

X_h_tr, X_h_val, y_h_tr, y_h_val = train_test_split(
    X_h, y_h, test_size=0.2, random_state=42, stratify=y_h)

n_tr_h = X_h_tr.shape[0]
rng = np.random.RandomState(42)
perm = rng.permutation(X_m.shape[0])
X_m_tr, y_m_tr = X_m[perm[:n_tr_h]], y_m[perm[:n_tr_h]]
X_m_val, y_m_val = X_m[perm[n_tr_h:]], y_m[perm[n_tr_h:]]

X_train = np.concatenate([X_h_tr, X_m_tr], axis=0)
y_train = np.concatenate([y_h_tr, y_m_tr], axis=0)
X_val   = np.concatenate([X_h_val, X_m_val], axis=0)
y_val   = np.concatenate([y_h_val, y_m_val], axis=0)

print("training sets:", X_train.shape, "HEALTHY:", (y_train==0).sum(), "MDD:", (y_train==1).sum())
print("validation sets:", X_val.shape,   "HEALTHY:", (y_val==0).sum(),   "MDD:", (y_val==1).sum())

In [ ]:
# 5. Dataset & DataLoader
import torch
from torch.utils.data import Dataset, DataLoader

class EEGCSPDataset(Dataset):
    def __init__(self, X, _, y):
        self.data = self._norm(X)
        self.labels = y
    def _norm(self, data):
        out = np.empty_like(data, dtype=np.float32)
        for i in range(data.shape[0]):
            s = data[i]
            mean = s.mean(axis=1, keepdims=True)
            std  = s.std(axis=1, keepdims=True)+1e-6
            out[i] = (s-mean)/std
        return out
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        x = torch.tensor(self.data[i])
        return x, torch.zeros(1), torch.tensor(self.labels[i])

BATCH_SIZE = 32
seed=42
g=torch.Generator().manual_seed(seed)
train_loader = DataLoader(EEGCSPDataset(X_train,None,y_train),
                          batch_size=BATCH_SIZE,shuffle=True,
                          worker_init_fn=lambda w: np.random.seed(seed+w),
                          generator=g)
val_loader   = DataLoader(EEGCSPDataset(X_val,  None,y_val),
                          batch_size=BATCH_SIZE,shuffle=False,
                          worker_init_fn=lambda w: np.random.seed(seed+w),
                          generator=g)

In [ ]:
# 6.  define ShallowConvNet 
import torch.nn as nn
import torch.nn.functional as F

class ShallowConvNet(nn.Module):
    def __init__(self, in_chans, n_classes, input_time_length, final_conv_length=None):
        super().__init__()
        # Time-Frequency Convolution
        self.conv_time = nn.Conv2d(1, 40, (1,25), bias=False)
        self.conv_spat = nn.Conv2d(40,40,(in_chans,1), bias=False)
        self.bn = nn.BatchNorm2d(40)
        self.pool = nn.AvgPool2d((1,75),(1,15))
        self.log = lambda x: torch.log(torch.clamp(x, min=1e-6))
        # Infer output length
        if final_conv_length is None:
            t = ((input_time_length-24)+1)
            t = (t-74)//15 + 1
        else:
            t = final_conv_length
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(40*t, n_classes)
        )
    def forward(self,x, _=None):
        # x: [B, C, T]
        x = x.unsqueeze(1)          # [B,1,C,T]
        x = self.conv_time(x)
        x = self.conv_spat(x)
        x = self.bn(x)
        x = torch.square(x)
        x = self.pool(x)
        x = self.log(x + 1e-6)
        return self.classifier(x)

# obtain indexes
batch = next(iter(train_loader))
inputs, _, labels = batch
in_chans = inputs.shape[1]
t_len    = inputs.shape[2]
n_classes= len(torch.unique(labels))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ShallowConvNet(in_chans, n_classes, t_len).to(device)

In [ ]:
# 7. Focal Loss + Optimizer
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.weight = weight
        self.reduction = reduction

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.weight)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss
    

# category weight 
cw = compute_class_weight('balanced',
                          classes=np.unique(y_train),
                          y=y_train)
cw = torch.tensor(cw, dtype=torch.float32).to(device)

criterion = FocalLoss(gamma=2.0, weight=cw)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [ ]:
# 8. training & validation loop 
import torch
from sklearn.metrics import recall_score, confusion_matrix, f1_score, roc_auc_score

num_epochs=200
train_losses, val_accs, sens, specs, f1s, aucs = [],[],[],[],[],[]

for ep in range(1, num_epochs+1):
    model.train()
    total_loss, cnt = 0,0
    for xb, _, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device).long()  
        optimizer.zero_grad()
        out = model(xb)
        loss= criterion(out, yb)
        loss.backward(); optimizer.step()
        total_loss += loss.item()*xb.size(0)
        cnt += xb.size(0)
    train_losses.append(total_loss/cnt)
    # val
    model.eval()
    y_true,y_pred,y_proba = [],[],[]
    with torch.no_grad():
        for xb, _, yb in val_loader:
            xb = xb.to(device)
            logits = model(xb)
            proba = F.softmax(logits, dim=1)[:,1].cpu().numpy()
            pred  = logits.argmax(1).cpu().numpy()
            y_true.extend(yb.numpy()); y_pred.extend(pred); y_proba.extend(proba)
    acc = np.mean(np.array(y_pred)==np.array(y_true))
    val_accs.append(acc)
    s = recall_score(y_true,y_pred,pos_label=1); sens.append(s)
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()
    specs.append(tn/(tn+fp))
    f1s.append(f1_score(y_true,y_pred,pos_label=1))
    try: aucs.append(roc_auc_score(y_true,y_proba))
    except: aucs.append(np.nan)

    print(f"Epoch {ep}/{num_epochs}  "
          f"Loss: {train_losses[-1]:.4f}  Acc: {acc:.4f}  "
          f"Sens: {s:.4f}  Spec: {specs[-1]:.4f}  "
          f"F1: {f1s[-1]:.4f}  AUC: {aucs[-1]:.4f}")



In [ ]:
# 9. save results & visualization
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter  # smooth curves
df_res = pd.DataFrame({
    'epoch':range(1,num_epochs+1),
    'train_loss':train_losses,
    'val_acc':val_accs,
    'sensitivity':sens,
    'specificity':specs,
    'f1_score':f1s,
    'auc':aucs
})
df_res.to_csv("training_shallow high gamma.csv", index=False, encoding="utf-8-sig")
print("all saved")
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei']
matplotlib.rcParams['axes.unicode_minus'] = False


import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.plot(savgol_filter(train_losses,11,3), label="smoothed")
plt.plot(train_losses, alpha=0.3, linestyle="--", label="original")
plt.title("Traning Loss"), plt.legend(), plt.grid(True)
plt.subplot(1,2,2)
plt.plot(savgol_filter(val_accs,11,3), label="smoothed")
plt.plot(val_accs, alpha=0.3, linestyle="--", label="original")
plt.title("Validation Accuracy"), plt.legend(), plt.grid(True)
plt.tight_layout(); plt.show()